# 穴埋め問題 (クローズテスト) を自動で作る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/25_gec_education/cloze_generation.ipynb)

NLP クックブック 25-19 ／ 文法誤り訂正と言語教育

文章から語を抜いて穴埋め問題を作り、もっともらしい誤答選択肢も自動で用意したい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import random
import torch
from fugashi import Tagger
from transformers import AutoModelForMaskedLM, AutoTokenizer

tagger = Tagger()
name = "tohoku-nlp/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForMaskedLM.from_pretrained(name).eval()

def make_cloze(sentence, target_pos=("名詞", "動詞", "形容詞"), n_distractors=3, seed=0):
    rng = random.Random(seed)
    words = [(w.surface, w.feature.pos1, w.feature.pos2) for w in tagger(sentence)]
    candidates = [i for i, (s, p1, p2) in enumerate(words) if p1 in target_pos and p2 not in ("非自立可能", "数詞") and len(tokenizer.tokenize(s)) == 1]
    if not candidates:
        return None
    i = rng.choice(candidates)
    answer, pos = words[i][0], words[i][1]
    masked = "".join(tokenizer.mask_token if j == i else s for j, (s, _, _) in enumerate(words))

    enc = tokenizer(masked, return_tensors="pt")
    mask_pos = (enc["input_ids"][0] == tokenizer.mask_token_id).nonzero()[0, 0]
    with torch.no_grad():
        probs = model(**enc).logits[0, mask_pos].softmax(-1)
    distractors = []
    for tok_id in probs.argsort(descending=True)[5:200].tolist():     # 上位すぎる語は「別の正解」になりうるので避ける
        tok = tokenizer.convert_ids_to_tokens(tok_id)
        analysis = list(tagger(tok))
        if len(analysis) == 1 and analysis[0].feature.pos1 == pos and tok != answer and not tok.startswith("##"):
            distractors.append(tok)
        if len(distractors) == n_distractors:
            break
    choices = distractors + [answer]
    rng.shuffle(choices)
    return {"question": masked.replace(tokenizer.mask_token, "（　　）"), "choices": choices, "answer": answer}

for s in ["太郎は図書館で本を借りた。", "雨が降ったので、試合は中止になった。", "この料理はとても辛いので、水をたくさん飲んだ。"]:
    q = make_cloze(s)
    print(q["question"], " / ".join(q["choices"]), f"(正解: {q['answer']})")

太郎は図書館で本を（　　）た。 開い / もらっ / 始め / 借り (正解: 借り)
雨が降ったので、試合は（　　）になった。 終了 / 休み / 無効 / 中止 (正解: 中止)
この料理はとても辛いので、水をたくさん（　　）だ。 持ち込ん / 選ん / 運ん / 飲ん (正解: 飲ん)


- 誤答選択肢は「同じ品詞で、文脈にそこそこ合うが、正解ではない語」が良い。マスク言語モデルの中位の候補はその性質に近い。
- 自動生成した選択肢には、正解になりうるものが混ざる。出題前に人間が確認する。
- 特定の文法項目 (助詞、活用) を問いたいときは、`target_pos` を変える。